# 08d Anchor-Offset Lane Postprocess v1

08c는 기울기 기억을 넣었지만, 영상상으로는 차가 **차선과의 간격을 유지한다**는 느낌이 약했다.

08d의 목표는 더 운전 직관에 가까운 구조다.

```text
lane detector output
→ lane geometry 해석
→ target path 생성
→ target path와의 간격 오차 + 방향 오차로 steer 계산
```

핵심 아이디어:

- 두 lane이 안정적으로 보이면 두 lane의 중앙선을 target path로 쓴다.
- 한 lane만 믿을 수 있으면 그 lane을 anchor로 삼고, 그 lane에서 일정 간격 떨어진 가상 target path를 만든다.
- 따라서 single lane 상황에서도 단순히 기울기만 따라가지 않고, lane과의 간격을 유지하려는 힘이 생긴다.
- lane이 모두 사라지면 직전 steer를 잠깐 유지하고, 오래 사라지면 저속으로 탐색한다.

이 노트북은 아직 최종 runtime 반영용이 아니라, **후처리 함수 설계와 field3 replay 영상 확인용**이다.


## 1. 상수와 경로

좌표계는 12번 실험 폴더 전체와 동일하다.

- 원본 frame: `1296 x 972`
- 학습/추론 preprocess: `cut_height=445`, resize to `800 x 320`, BGR `/255`
- decoder output lane points: 원본 frame 좌표계의 `(x, y)` points

08d는 decoder 이후 좌표만 받기 때문에 모델/ONNX에는 관여하지 않는다.


In [ ]:
from __future__ import annotations

import csv
import json
import math
from collections import Counter
from pathlib import Path

import cv2
import numpy as np

BASE = Path(r"~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild")
REVIEW_ROOT = BASE / "review_outputs" / "08d_anchor_offset_lane_postprocess_v1"
VIDEO_DIR = REVIEW_ROOT / "videos"
TABLE_DIR = REVIEW_ROOT / "tables"
CONFIG_DIR = REVIEW_ROOT / "config"
for d in [REVIEW_ROOT, VIDEO_DIR, TABLE_DIR, CONFIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PKG10 = BASE / "review_outputs" / "10_pi_runtime_latency_sequence_validation_v1" / "pkg"
RECORDS_CSV = PKG10 / "t" / "records_manifest.csv"
DECODED_JSONL = PKG10 / "r" / "ref_decoded.jsonl"

RAW_W = 1296
RAW_H = 972
CUT_HEIGHT = 445
IMAGE_CENTER_X = RAW_W / 2.0
HALF_W = RAW_W / 2.0

FIELD3_VIDEO_PATH = VIDEO_DIR / "field3_08d_anchor_offset_lane_postprocess_v1.mp4"
FIELD3_SEQUENCE_CSV = TABLE_DIR / "field3_08d_anchor_offset_lane_postprocess_v1.csv"
CONFIG_JSON = CONFIG_DIR / "lane_behavior_08d_anchor_offset_v1.json"

print("review root:", REVIEW_ROOT)
print("records:", RECORDS_CSV)
print("decoded:", DECODED_JSONL)


## 2. 사람이 조절할 tuning config

px 단위 기준은 가능하면 아래에 노출하지 않는다.  
현장에서 수정해야 할 가능성이 높은 값만 행동 단위로 둔다.

- `single_offset_ratio`: 한쪽 lane만 믿을 때, 원래 중앙선까지 떨어지지 않고 anchor lane에 얼마나 붙을지 결정한다.
  - `1.0`: 정상 중앙선 위치까지 떨어져 달림
  - `0.7`: anchor lane에 더 붙어서 달림
  - `0.5`: 더 바짝 붙음
- `center_gain`: target path와 차 중심의 x 오차를 줄이는 힘
- `heading_gain`: target path가 향하는 방향을 따라가는 힘
- `anchor_*`: single/anchor mode에서 조금 더 조심하거나 강하게 돌게 하는 gain
- `speed_scale`: lane 상황별 속도 scale. 실제 motor base speed는 runtime에서 곱해진다.


In [ ]:
LANE_BEHAVIOR_08D = {
    # y anchor. near는 차 바로 앞, far는 전방 진행 방향 판단에 사용한다.
    "near_y_ratio": 0.96,
    "mid_y_ratio": 0.84,
    "far_y_ratio": 0.68,

    # 두 lane 중앙선 기반 조향.
    "both_center_gain": 1.00,
    "both_heading_gain": 0.45,

    # 한 lane anchor 기반 조향.
    # single_offset_ratio가 작을수록 anchor lane에 더 붙어서 돈다.
    "single_offset_ratio": 0.70,
    "anchor_center_gain": 1.15,
    "anchor_heading_gain": 0.75,

    # memory. stable pair에서 얻은 lane gap/heading을 천천히 갱신한다.
    "memory_alpha": 0.25,
    "steer_alpha_both": 0.60,
    "steer_alpha_anchor": 0.45,

    # lost handling.
    "lost_hold_frames": 4,
    "lost_search_boost": 1.10,
    "lost_decay": 0.92,

    # mode별 속도 scale. 코너/불안정 구간에서 lane postprocess가 직접 감속 의도를 낸다.
    "speed_both": 1.00,
    "speed_anchor": 0.55,
    "speed_lost_hold": 0.35,
    "speed_lost_search": 0.25,

    "max_steer_norm": 0.75,
}

print(json.dumps(LANE_BEHAVIOR_08D, indent=2, ensure_ascii=False))


## 3. 내부 threshold 자동 계산

아래 값들은 px 단위지만 사람이 직접 튜닝하지 않게 내부에서 계산한다.

- lane point가 y anchor 주변에 충분히 있어야 feature로 인정한다.
- pair gap은 원본 frame 폭 기준 비율로 판단한다.
- 이전 stable pair와 비교해서 한쪽 lane만 안정적인 경우 anchor mode로 보낸다.


In [ ]:
def clamp(v, lo, hi):
    return max(lo, min(hi, v))


def lerp(a, b, alpha):
    return (1.0 - alpha) * float(a) + alpha * float(b)


def derive_internal_thresholds(cfg):
    near_y = RAW_H * float(cfg["near_y_ratio"])
    mid_y = RAW_H * float(cfg["mid_y_ratio"])
    far_y = RAW_H * float(cfg["far_y_ratio"])
    return {
        "near_y": near_y,
        "mid_y": mid_y,
        "far_y": far_y,
        "min_points": 4,
        "min_y_span_px": RAW_H * 0.06,
        "max_interp_gap_px": RAW_H * 0.14,
        "pair_gap_min_px": RAW_W * 0.20,
        "pair_gap_max_px": RAW_W * 0.95,
        "fallback_half_gap_px": RAW_W * 0.28,
        "anchor_x_stable_px": RAW_W * 0.18,
        "anchor_heading_stable": 0.30,
        "center_jump_norm": 0.28,
    }

INTERNAL_08D = derive_internal_thresholds(LANE_BEHAVIOR_08D)
print(json.dumps(INTERNAL_08D, indent=2, ensure_ascii=False))


## 4. Lane feature 추출

각 lane polyline에서 `near/mid/far` x를 읽는다.

```text
x_far  : 전방 방향 판단
x_mid  : 조향 target 위치 판단
x_near : 현재 차 가까운 쪽 위치 판단
heading = (x_far - x_near) / half_screen_width
```

`heading > 0`이면 target path가 화면 오른쪽으로 향한다는 뜻이고, 최종 steer도 오른쪽 방향으로 양수다.


In [ ]:
def interp_or_nearest_x(points, y, max_gap_px):
    pts = np.asarray(points, dtype=np.float32)
    if pts.ndim != 2 or pts.shape[1] != 2 or len(pts) < 2:
        return None
    pts = pts[np.isfinite(pts).all(axis=1)]
    if len(pts) < 2:
        return None

    order = np.argsort(pts[:, 1])
    xs = pts[order, 0]
    ys = pts[order, 1]

    if y < ys[0] or y > ys[-1]:
        return None

    idx = int(np.searchsorted(ys, y))
    if idx <= 0:
        nearest_gap = abs(float(ys[0] - y))
        return float(xs[0]) if nearest_gap <= max_gap_px else None
    if idx >= len(ys):
        nearest_gap = abs(float(ys[-1] - y))
        return float(xs[-1]) if nearest_gap <= max_gap_px else None

    y0, y1 = float(ys[idx - 1]), float(ys[idx])
    x0, x1 = float(xs[idx - 1]), float(xs[idx])
    if abs(y1 - y0) > max_gap_px:
        return None
    if abs(y1 - y0) < 1e-6:
        return x0
    t = (float(y) - y0) / (y1 - y0)
    return x0 + t * (x1 - x0)


def lane_feature(lane, internal):
    points = np.asarray(lane.get("points", []), dtype=np.float32)
    if points.ndim != 2 or points.shape[1] != 2:
        return None
    if len(points) < int(internal["min_points"]):
        return None
    y_span = float(np.nanmax(points[:, 1]) - np.nanmin(points[:, 1]))
    if y_span < float(internal["min_y_span_px"]):
        return None

    x_near = interp_or_nearest_x(points, internal["near_y"], internal["max_interp_gap_px"])
    x_mid = interp_or_nearest_x(points, internal["mid_y"], internal["max_interp_gap_px"])
    x_far = interp_or_nearest_x(points, internal["far_y"], internal["max_interp_gap_px"])
    if x_near is None or x_mid is None or x_far is None:
        return None

    heading = (float(x_far) - float(x_near)) / HALF_W
    return {
        "x_near": float(x_near),
        "x_mid": float(x_mid),
        "x_far": float(x_far),
        "heading": float(heading),
        "conf": float(lane.get("conf", 0.0)),
        "y_span": y_span,
        "raw_lane": lane,
    }


def extract_lane_features(lanes, internal):
    feats = []
    for lane in lanes:
        f = lane_feature(lane, internal)
        if f is not None:
            feats.append(f)
    feats.sort(key=lambda f: f["x_mid"])
    return feats

print("lane feature helpers ready")


## 5. Memory

08d의 memory는 단순히 이전 steer만 기억하지 않는다.

stable both-lane 상황에서 다음을 기억한다.

- left/right lane의 `x_mid`
- lane gap의 절반값 `half_gap`
- 중앙 target path의 `target_near/mid/far`
- 중앙선 heading

single lane 상황에서는 이 memory를 이용해서 현재 lane이 left anchor인지 right anchor인지 판단하고, 그 lane에서 어느 정도 떨어진 target path를 만든다.


In [ ]:
def init_memory_08d():
    return {
        "has_stable_pair": False,
        "left_mid": None,
        "right_mid": None,
        "half_gap_near": INTERNAL_08D["fallback_half_gap_px"],
        "half_gap_mid": INTERNAL_08D["fallback_half_gap_px"],
        "half_gap_far": INTERNAL_08D["fallback_half_gap_px"],
        "target_near": IMAGE_CENTER_X,
        "target_mid": IMAGE_CENTER_X,
        "target_far": IMAGE_CENTER_X,
        "heading": 0.0,
        "last_steer_norm": 0.0,
        "last_mode": "init",
        "lost_frames": 0,
    }


def update_pair_memory(memory, geom, cfg):
    alpha = float(cfg["memory_alpha"])
    if not memory["has_stable_pair"]:
        alpha = 1.0

    memory["has_stable_pair"] = True
    memory["left_mid"] = lerp(memory["left_mid"] if memory["left_mid"] is not None else geom["left"]["x_mid"], geom["left"]["x_mid"], alpha)
    memory["right_mid"] = lerp(memory["right_mid"] if memory["right_mid"] is not None else geom["right"]["x_mid"], geom["right"]["x_mid"], alpha)
    memory["half_gap_near"] = lerp(memory["half_gap_near"], geom["half_gap_near"], alpha)
    memory["half_gap_mid"] = lerp(memory["half_gap_mid"], geom["half_gap_mid"], alpha)
    memory["half_gap_far"] = lerp(memory["half_gap_far"], geom["half_gap_far"], alpha)
    memory["target_near"] = lerp(memory["target_near"], geom["target_near"], alpha)
    memory["target_mid"] = lerp(memory["target_mid"], geom["target_mid"], alpha)
    memory["target_far"] = lerp(memory["target_far"], geom["target_far"], alpha)
    memory["heading"] = lerp(memory["heading"], geom["heading"], alpha)

print("memory helpers ready")


## 6. Pair와 anchor 선택

### Pair
두 lane이 보이면 가능한 pair들을 만들고, gap과 confidence를 기준으로 가장 그럴듯한 pair를 선택한다.

### Anchor
pair가 없거나, pair 내부에서 한쪽 lane만 이전 stable geometry와 일관되면 그 lane을 anchor로 쓴다.

anchor mode의 target path:

```text
left anchor : target_x = left_lane_x + remembered_half_gap * single_offset_ratio
right anchor: target_x = right_lane_x - remembered_half_gap * single_offset_ratio
```

즉 한쪽 lane만 보일 때도, 그 lane과의 목표 간격을 유지하려고 한다.


In [ ]:
def pair_geometry(left, right):
    # left/right는 x_mid 기준으로 정렬된 feature여야 한다.
    target_near = 0.5 * (left["x_near"] + right["x_near"])
    target_mid = 0.5 * (left["x_mid"] + right["x_mid"])
    target_far = 0.5 * (left["x_far"] + right["x_far"])
    return {
        "left": left,
        "right": right,
        "target_near": float(target_near),
        "target_mid": float(target_mid),
        "target_far": float(target_far),
        "half_gap_near": float(0.5 * abs(right["x_near"] - left["x_near"])),
        "half_gap_mid": float(0.5 * abs(right["x_mid"] - left["x_mid"])),
        "half_gap_far": float(0.5 * abs(right["x_far"] - left["x_far"])),
        "heading": float((target_far - target_near) / HALF_W),
        "conf": float(0.5 * (left["conf"] + right["conf"])),
    }


def valid_pair(g, internal):
    gap_mid = 2.0 * float(g["half_gap_mid"])
    return internal["pair_gap_min_px"] <= gap_mid <= internal["pair_gap_max_px"]


def pair_score(g, memory):
    # 중앙에 너무 치우치지 않고, confidence가 높고, 이전 center와 크게 다르지 않은 pair 선호.
    center_norm = abs((g["target_mid"] - IMAGE_CENTER_X) / HALF_W)
    score = float(g["conf"]) - 0.30 * center_norm
    if memory["has_stable_pair"]:
        mem_center_norm = abs((g["target_mid"] - memory["target_mid"]) / HALF_W)
        mem_heading_diff = abs(g["heading"] - memory["heading"])
        score -= 0.25 * mem_center_norm + 0.15 * mem_heading_diff
    return score


def best_pair(features, memory, internal):
    if len(features) < 2:
        return None
    best = None
    best_score = -1e9
    for i in range(len(features)):
        for j in range(i + 1, len(features)):
            left, right = features[i], features[j]
            if left["x_mid"] >= right["x_mid"]:
                continue
            g = pair_geometry(left, right)
            if not valid_pair(g, internal):
                continue
            s = pair_score(g, memory)
            if s > best_score:
                best = g
                best_score = s
    return best


def lane_stability_cost(feature, role, memory):
    if not memory["has_stable_pair"]:
        # memory가 없으면 화면 위치만으로 role을 추정한다.
        expected_x = IMAGE_CENTER_X - INTERNAL_08D["fallback_half_gap_px"] if role == "left" else IMAGE_CENTER_X + INTERNAL_08D["fallback_half_gap_px"]
        expected_heading = 0.0
    else:
        expected_x = memory["left_mid"] if role == "left" else memory["right_mid"]
        expected_heading = memory["heading"]
    x_cost = abs(float(feature["x_mid"]) - float(expected_x)) / HALF_W
    h_cost = abs(float(feature["heading"]) - float(expected_heading))
    conf_bonus = 0.15 * float(feature["conf"])
    return x_cost + 0.70 * h_cost - conf_bonus


def choose_anchor_feature(features, memory):
    if not features:
        return None, None, None
    candidates = []
    for f in features:
        for role in ["left", "right"]:
            candidates.append((lane_stability_cost(f, role, memory), f, role))
    candidates.sort(key=lambda x: x[0])
    cost, feature, role = candidates[0]
    return feature, role, float(cost)


def pair_has_one_stable_lane(pair, memory, internal):
    if pair is None or not memory["has_stable_pair"]:
        return None
    left = pair["left"]
    right = pair["right"]
    left_dx = abs(left["x_mid"] - memory["left_mid"])
    right_dx = abs(right["x_mid"] - memory["right_mid"])
    left_dh = abs(left["heading"] - memory["heading"])
    right_dh = abs(right["heading"] - memory["heading"])

    left_stable = left_dx <= internal["anchor_x_stable_px"] and left_dh <= internal["anchor_heading_stable"]
    right_stable = right_dx <= internal["anchor_x_stable_px"] and right_dh <= internal["anchor_heading_stable"]
    center_jump = abs((pair["target_mid"] - memory["target_mid"]) / HALF_W) >= internal["center_jump_norm"]

    if center_jump and left_stable and not right_stable:
        return left, "left", "right_lane_jump"
    if center_jump and right_stable and not left_stable:
        return right, "right", "left_lane_jump"
    return None


def anchor_target_path(feature, role, memory, cfg):
    ratio = float(cfg["single_offset_ratio"])
    sign = 1.0 if role == "left" else -1.0
    target_near = feature["x_near"] + sign * memory["half_gap_near"] * ratio
    target_mid = feature["x_mid"] + sign * memory["half_gap_mid"] * ratio
    target_far = feature["x_far"] + sign * memory["half_gap_far"] * ratio
    heading = (target_far - target_near) / HALF_W
    return {
        "target_near": float(target_near),
        "target_mid": float(target_mid),
        "target_far": float(target_far),
        "heading": float(heading),
        "anchor": feature,
        "anchor_role": role,
        "conf": float(feature["conf"]),
    }

print("pair/anchor helpers ready")


## 7. Geometry → drive command

08d의 drive command는 다음 두 오차로 만든다.

```text
center_error  = target_mid가 화면 중앙에서 얼마나 벗어났는가
heading_error = target path가 near→far로 얼마나 좌/우를 향하는가
steer = center_gain * center_error + heading_gain * heading_error
```

한쪽 lane anchor mode에서는 같은 식을 쓰되, gain과 target path가 달라진다.


In [ ]:
def clip_steer(v, cfg):
    m = float(cfg["max_steer_norm"])
    return float(clamp(v, -m, m))


def target_to_command(target, mode, memory, cfg):
    center_error = (float(target["target_mid"]) - IMAGE_CENTER_X) / HALF_W
    heading_error = float(target["heading"])

    if mode == "both_center":
        raw = float(cfg["both_center_gain"]) * center_error + float(cfg["both_heading_gain"]) * heading_error
        alpha = float(cfg["steer_alpha_both"])
        speed_scale = float(cfg["speed_both"])
    elif mode == "anchor_follow":
        raw = float(cfg["anchor_center_gain"]) * center_error + float(cfg["anchor_heading_gain"]) * heading_error
        alpha = float(cfg["steer_alpha_anchor"])
        speed_scale = float(cfg["speed_anchor"])
    else:
        raise ValueError(mode)

    raw = clip_steer(raw, cfg)
    steer = clip_steer(lerp(memory["last_steer_norm"], raw, alpha), cfg)
    return {
        "raw_steer_norm": float(raw),
        "steer_norm": float(steer),
        "speed_scale": float(speed_scale),
        "center_error": float(center_error),
        "heading_error": float(heading_error),
    }


def update_drive_08d(lanes, memory, cfg=LANE_BEHAVIOR_08D):
    internal = derive_internal_thresholds(cfg)
    features = extract_lane_features(lanes, internal)
    pair = best_pair(features, memory, internal)

    reason = ""
    target = None
    mode = None

    # Pair는 보이지만 한쪽이 튄 경우 stable lane을 anchor로 선택한다.
    one_stable = pair_has_one_stable_lane(pair, memory, internal)
    if one_stable is not None:
        feature, role, reason = one_stable
        target = anchor_target_path(feature, role, memory, cfg)
        mode = "anchor_follow"
    elif pair is not None:
        target = pair
        mode = "both_center"
        reason = "valid_pair"
    elif features:
        feature, role, cost = choose_anchor_feature(features, memory)
        target = anchor_target_path(feature, role, memory, cfg)
        mode = "anchor_follow"
        reason = f"single_or_no_valid_pair:{role}:cost={cost:.3f}"
    else:
        memory["lost_frames"] += 1
        if memory["lost_frames"] <= int(cfg["lost_hold_frames"]):
            steer = clip_steer(memory["last_steer_norm"], cfg)
            mode = "lost_hold"
            speed_scale = float(cfg["speed_lost_hold"])
            reason = "no_feature_hold_last_steer"
        else:
            steer = clip_steer(memory["last_steer_norm"] * float(cfg["lost_search_boost"]), cfg)
            steer = clip_steer(steer * float(cfg["lost_decay"]), cfg)
            mode = "lost_search"
            speed_scale = float(cfg["speed_lost_search"])
            reason = "no_feature_search_previous_direction"
        memory["last_steer_norm"] = steer
        memory["last_mode"] = mode
        return {
            "mode": mode,
            "steer_norm": float(steer),
            "raw_steer_norm": float(steer),
            "speed_scale": float(speed_scale),
            "center_error": 0.0,
            "heading_error": 0.0,
            "target_near": float(memory["target_near"]),
            "target_mid": float(memory["target_mid"]),
            "target_far": float(memory["target_far"]),
            "confidence": 0.0,
            "lane_count": len(lanes),
            "feature_count": len(features),
            "lost_frames": int(memory["lost_frames"]),
            "anchor_role": "none",
            "reason": reason,
        }

    cmd = target_to_command(target, mode, memory, cfg)
    memory["lost_frames"] = 0

    if mode == "both_center":
        update_pair_memory(memory, target, cfg)
        anchor_role = "pair"
        confidence = float(target["conf"])
    else:
        # anchor mode는 안정 pair memory 자체는 갱신하지 않는다.
        # 다만 target path와 heading은 마지막 참조로 기억한다.
        memory["target_near"] = float(target["target_near"])
        memory["target_mid"] = float(target["target_mid"])
        memory["target_far"] = float(target["target_far"])
        memory["heading"] = float(target["heading"])
        anchor_role = str(target["anchor_role"])
        confidence = float(target["conf"])

    memory["last_steer_norm"] = float(cmd["steer_norm"])
    memory["last_mode"] = mode

    return {
        "mode": mode,
        "steer_norm": float(cmd["steer_norm"]),
        "raw_steer_norm": float(cmd["raw_steer_norm"]),
        "speed_scale": float(cmd["speed_scale"]),
        "center_error": float(cmd["center_error"]),
        "heading_error": float(cmd["heading_error"]),
        "target_near": float(target["target_near"]),
        "target_mid": float(target["target_mid"]),
        "target_far": float(target["target_far"]),
        "confidence": confidence,
        "lane_count": len(lanes),
        "feature_count": len(features),
        "lost_frames": int(memory["lost_frames"]),
        "anchor_role": anchor_role,
        "reason": reason,
    }

print("08d drive command ready")


## 8. 간단 synthetic check

실제 field3 영상 전에, 수식 방향이 이상하지 않은지 확인한다.

- 직선 pair: steer가 0 근처
- 오른쪽으로 향하는 pair: steer 양수
- 왼쪽 anchor: anchor lane에서 offset된 target path를 만들고 조향
- lost: 이전 steer 유지


In [ ]:
def make_line_lane(x_near, x_far, conf=0.9):
    ys = np.linspace(RAW_H - 1, RAW_H * 0.60, 12, dtype=np.float32)
    xs = np.linspace(float(x_near), float(x_far), len(ys), dtype=np.float32)
    return {"points": np.stack([xs, ys], axis=1), "conf": float(conf)}

memory = init_memory_08d()
examples = [
    ("straight_pair", [make_line_lane(360, 360), make_line_lane(940, 940)]),
    ("right_pair", [make_line_lane(360, 430), make_line_lane(940, 1010)]),
    ("left_anchor_only", [make_line_lane(330, 260)]),
    ("lost", []),
]
for name, lanes in examples:
    d = update_drive_08d(lanes, memory, LANE_BEHAVIOR_08D)
    print(name, {k: d[k] for k in ["mode", "anchor_role", "steer_norm", "speed_scale", "center_error", "heading_error", "reason"]})


## 9. field3 replay helpers

10번 Pi runtime validation package에 저장된 field3 decoded lane을 재사용한다.  
즉, 여기서 보는 것은 모델 성능이 아니라 **08d postprocess가 decoder lane을 어떻게 해석하는지**다.


In [ ]:
def imread_bgr_unicode(path):
    path = Path(path)
    data = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return img


def load_field3_sequence_records(limit=None):
    rows = []
    with RECORDS_CSV.open("r", encoding="utf-8-sig", newline="") as f:
        for row in csv.DictReader(f):
            if row["set"] == "field3" and row["role"] == "sequence":
                row["order"] = int(row["order"])
                rows.append(row)
    rows.sort(key=lambda r: r["order"])
    if limit is not None:
        rows = rows[: int(limit)]
    return rows


def load_decoded_lanes_by_key():
    out = {}
    with DECODED_JSONL.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            out[obj["key"]] = obj.get("lanes", [])
    return out


def lanes_from_jsonable(lanes_json):
    lanes = []
    for lane in lanes_json:
        pts = np.asarray(lane.get("points", []), dtype=np.float32)
        if pts.ndim != 2 or pts.shape[1] != 2 or len(pts) < 2:
            continue
        lanes.append({"points": pts, "conf": float(lane.get("conf", 0.0))})
    return lanes


def draw_lane_polyline(img, points, color=(0, 220, 80), thickness=3):
    pts = np.asarray(points, dtype=np.float32)
    valid = np.isfinite(pts).all(axis=1)
    pts = pts[valid]
    if len(pts) < 2:
        return
    pts_i = np.round(pts).astype(np.int32).reshape(-1, 1, 2)
    cv2.polylines(img, [pts_i], False, color, thickness, cv2.LINE_AA)


def draw_text_lines(img, lines, x=24, y=34, line_h=28):
    pad = 10
    width = max(620, max((len(s) for s in lines), default=0) * 12)
    height = line_h * len(lines) + pad * 2
    panel = img.copy()
    cv2.rectangle(panel, (x - pad, y - 24), (x - pad + width, y - 24 + height), (0, 0, 0), -1)
    cv2.addWeighted(panel, 0.55, img, 0.45, 0, img)
    for i, text in enumerate(lines):
        cv2.putText(img, text, (x, y + i * line_h), cv2.FONT_HERSHEY_SIMPLEX, 0.70, (245, 245, 245), 2, cv2.LINE_AA)

print("field3 IO helpers ready")


## 10. field3 overlay drawing

영상에서 볼 것:

- green: decoder lane
- cyan line: 08d가 만든 target path
- magenta arrow: 최종 steer
- orange dot/line: target midpoint와 center error
- text: mode, anchor role, steer, speed scale, heading, reason


In [ ]:
def draw_08d_replay_frame(bgr, lanes, drive, frame_index, cfg=LANE_BEHAVIOR_08D, scale_width=960):
    out = bgr.copy()
    internal = derive_internal_thresholds(cfg)

    for lane in lanes:
        draw_lane_polyline(out, lane["points"], color=(0, 220, 80), thickness=4)

    for name, color in [("far", (255, 220, 0)), ("mid", (255, 180, 0)), ("near", (255, 140, 0))]:
        y = int(round(internal[f"{name}_y"]))
        cv2.line(out, (0, y), (RAW_W - 1, y), color, 1, cv2.LINE_AA)
        cv2.putText(out, name, (12, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)

    # center reference
    cv2.line(out, (int(IMAGE_CENTER_X), int(CUT_HEIGHT)), (int(IMAGE_CENTER_X), RAW_H - 1), (210, 210, 210), 2, cv2.LINE_AA)

    # target path from near to far
    p_near = (int(round(drive["target_near"])), int(round(internal["near_y"])))
    p_mid = (int(round(drive["target_mid"])), int(round(internal["mid_y"])))
    p_far = (int(round(drive["target_far"])), int(round(internal["far_y"])))
    cv2.line(out, p_near, p_mid, (255, 255, 0), 4, cv2.LINE_AA)
    cv2.line(out, p_mid, p_far, (255, 255, 0), 4, cv2.LINE_AA)
    cv2.circle(out, p_mid, 9, (0, 170, 255), -1, cv2.LINE_AA)
    cv2.line(out, (int(IMAGE_CENTER_X), p_mid[1]), p_mid, (0, 170, 255), 3, cv2.LINE_AA)

    # steer command arrow
    base = (int(IMAGE_CENTER_X), RAW_H - 35)
    steer_tip = (int(round(IMAGE_CENTER_X + float(drive["steer_norm"]) * 420.0)), int(round(internal["far_y"])))
    cv2.arrowedLine(out, base, steer_tip, (255, 0, 220), 5, cv2.LINE_AA, tipLength=0.20)

    # raw steer thin arrow
    raw_tip = (int(round(IMAGE_CENTER_X + float(drive["raw_steer_norm"]) * 420.0)), int(round(internal["mid_y"])))
    cv2.arrowedLine(out, base, raw_tip, (180, 120, 180), 2, cv2.LINE_AA, tipLength=0.18)

    mode = str(drive["mode"])
    mode_color = {
        "both_center": (80, 220, 80),
        "anchor_follow": (0, 220, 255),
        "lost_hold": (0, 165, 255),
        "lost_search": (0, 60, 255),
    }.get(mode, (255, 255, 255))
    cv2.circle(out, (RAW_W - 34, 34), 16, mode_color, -1, cv2.LINE_AA)

    lines = [
        f"{frame_index:04d} mode={mode} anchor={drive['anchor_role']} lanes={drive['lane_count']} feat={drive['feature_count']}",
        f"steer={drive['steer_norm']:+.3f} raw={drive['raw_steer_norm']:+.3f} speed={drive['speed_scale']:.2f} conf={drive['confidence']:.2f}",
        f"center={drive['center_error']:+.3f} heading={drive['heading_error']:+.3f} lost={drive['lost_frames']}",
        f"reason={drive['reason']}",
    ]
    draw_text_lines(out, lines)

    if scale_width is not None and scale_width > 0 and out.shape[1] != scale_width:
        scale = float(scale_width) / float(out.shape[1])
        out = cv2.resize(out, (scale_width, int(round(out.shape[0] * scale))), interpolation=cv2.INTER_AREA)
    return out

print("overlay helpers ready")


## 11. field3 replay video 생성

240-frame field3 sequence 전체를 순서대로 돌린다.  
memory가 누적되므로 단일 frame overlay보다 실제 주행 후처리 판단에 더 가깝다.


In [ ]:
def generate_field3_08d_video(limit=None, fps=12.0):
    CONFIG_JSON.write_text(json.dumps(LANE_BEHAVIOR_08D, indent=2, ensure_ascii=False), encoding="utf-8")

    records = load_field3_sequence_records(limit=limit)
    decoded_by_key = load_decoded_lanes_by_key()
    memory = init_memory_08d()
    writer = None
    rows = []

    for idx, rec in enumerate(records):
        bgr = imread_bgr_unicode(PKG10 / rec["image_rel"])
        lanes = lanes_from_jsonable(decoded_by_key.get(rec["key"], []))
        drive = update_drive_08d(lanes, memory, LANE_BEHAVIOR_08D)
        frame = draw_08d_replay_frame(bgr, lanes, drive, idx, LANE_BEHAVIOR_08D, scale_width=960)

        if writer is None:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(str(FIELD3_VIDEO_PATH), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
            assert writer.isOpened(), FIELD3_VIDEO_PATH

        writer.write(frame)
        rows.append({
            "frame_index": idx,
            "key": rec["key"],
            "source_name": rec["source_name"],
            "mode": drive["mode"],
            "anchor_role": drive["anchor_role"],
            "lane_count": drive["lane_count"],
            "feature_count": drive["feature_count"],
            "steer_norm": float(drive["steer_norm"]),
            "raw_steer_norm": float(drive["raw_steer_norm"]),
            "speed_scale": float(drive["speed_scale"]),
            "center_error": float(drive["center_error"]),
            "heading_error": float(drive["heading_error"]),
            "target_near": float(drive["target_near"]),
            "target_mid": float(drive["target_mid"]),
            "target_far": float(drive["target_far"]),
            "confidence": float(drive["confidence"]),
            "lost_frames": int(drive["lost_frames"]),
            "reason": drive["reason"],
        })

    if writer is not None:
        writer.release()

    with FIELD3_SEQUENCE_CSV.open("w", encoding="utf-8-sig", newline="") as f:
        writer_csv = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else ["frame_index"])
        writer_csv.writeheader()
        writer_csv.writerows(rows)

    mode_counts = Counter(row["mode"] for row in rows)
    anchor_counts = Counter(row["anchor_role"] for row in rows)
    steer_abs = np.asarray([abs(row["steer_norm"]) for row in rows], dtype=np.float32)
    raw_abs = np.asarray([abs(row["raw_steer_norm"]) for row in rows], dtype=np.float32)
    speed = np.asarray([row["speed_scale"] for row in rows], dtype=np.float32)
    summary = {
        "frames": len(rows),
        "video": str(FIELD3_VIDEO_PATH),
        "table": str(FIELD3_SEQUENCE_CSV),
        "config": str(CONFIG_JSON),
        "mode_counts": dict(mode_counts),
        "anchor_counts": dict(anchor_counts),
        "mean_abs_steer": float(steer_abs.mean()) if len(steer_abs) else None,
        "p90_abs_steer": float(np.percentile(steer_abs, 90)) if len(steer_abs) else None,
        "max_abs_steer": float(steer_abs.max()) if len(steer_abs) else None,
        "p90_abs_raw_steer": float(np.percentile(raw_abs, 90)) if len(raw_abs) else None,
        "mean_speed_scale": float(speed.mean()) if len(speed) else None,
        "min_speed_scale": float(speed.min()) if len(speed) else None,
    }
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    return summary

field3_08d_video_summary = generate_field3_08d_video(limit=None, fps=12.0)


## 12. 읽을 때 볼 포인트

영상에서 다음을 본다.

1. `anchor_follow`에서 target path가 초록 lane에 적당히 붙어서 만들어지는가?
2. 코너에서 `steer_norm`이 08c보다 충분히 강하게 나오는가?
3. `both_center`로 돌아왔을 때 target path가 즉시 중앙선으로 복귀하는가?
4. `lost_hold`가 너무 많이 나오거나, 너무 오래 유지되지 않는가?
5. speed scale이 불안정 구간에서 충분히 낮아지는가?

마음에 안 드는 경우 가장 먼저 볼 값은 다음 네 개다.

```python
single_offset_ratio
anchor_center_gain
anchor_heading_gain
steer_alpha_anchor
```
